In [29]:
import pandas as pd

df = pd.read_csv('gk_qna_dataset.csv')

df.head()

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo


In [30]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [31]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [32]:
# vocab
vocab = {'<UNK>':0}

In [33]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)


In [34]:
df.apply(build_vocab, axis=1)

0      None
1      None
2      None
3      None
4      None
       ... 
166    None
167    None
168    None
169    None
170    None
Length: 171, dtype: object

In [35]:
len(vocab)

285

In [36]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [37]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

In [39]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [40]:
dataset = QADataset(df, vocab)

In [41]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [42]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[248, 249, 250, 239,  85,   3, 281]]) tensor([167, 168])
tensor([[ 1,  2,  3,  4,  5, 12]]) tensor([13])
tensor([[252,   1, 253, 254, 255, 199,   2, 271]]) tensor([203])
tensor([[ 80, 171, 175, 176]]) tensor([177, 178, 179])
tensor([[246,   2,   3, 112, 247, 242,  96, 243, 244]]) tensor([114, 113])
tensor([[252,   1, 253, 254, 239, 171, 175, 273]]) tensor([177, 178, 179])
tensor([[237, 238, 246, 132, 140, 134, 268]]) tensor([141, 131, 136])
tensor([[252,   1, 253, 254, 239,  85,   3, 257]]) tensor([87, 88, 89])
tensor([[262, 243, 263, 264, 239, 171, 261]]) tensor([173, 174])
tensor([[248, 249, 250, 246, 132, 133, 134, 268]]) tensor([135, 131, 136])
tensor([[237, 238, 246,   2,   3, 222, 130,   5, 266]]) tensor([224])
tensor([[ 94,  95,  96, 208, 209]]) tensor([210])
tensor([[ 96, 243, 245, 246,   2,  90,  91, 282]]) tensor([93])
tensor([[262, 243, 263, 264, 246,   2,   3, 232, 185,   5, 265]]) tensor([234, 235, 236])
tensor([[262, 243, 263, 264, 246,   2, 120, 121, 268]]) tenso

In [43]:
import torch.nn as nn

In [44]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [45]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [46]:
learning_rate = 0.001
epochs = 200

In [47]:
model = SimpleRNN(len(vocab))

In [48]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [49]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (batch, vocab_size) and target shape (batch,)
    # use the first token of the answer sequence as the target to match batch dimension
    loss = criterion(output, answer[:, 0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 906.053792
Epoch: 2, Loss: 605.113288
Epoch: 3, Loss: 431.469084
Epoch: 4, Loss: 322.836261
Epoch: 5, Loss: 246.285541
Epoch: 6, Loss: 189.876845
Epoch: 7, Loss: 145.863523
Epoch: 8, Loss: 112.067316
Epoch: 9, Loss: 84.888388
Epoch: 10, Loss: 63.090073
Epoch: 11, Loss: 47.204001
Epoch: 12, Loss: 35.401599
Epoch: 13, Loss: 26.308414
Epoch: 14, Loss: 20.042710
Epoch: 15, Loss: 15.797708
Epoch: 16, Loss: 12.831657
Epoch: 17, Loss: 10.528295
Epoch: 18, Loss: 8.862626
Epoch: 19, Loss: 7.496767
Epoch: 20, Loss: 6.418528
Epoch: 21, Loss: 5.545181
Epoch: 22, Loss: 4.806013
Epoch: 23, Loss: 4.200998
Epoch: 24, Loss: 3.700661
Epoch: 25, Loss: 3.267978
Epoch: 26, Loss: 2.890741
Epoch: 27, Loss: 2.573034
Epoch: 28, Loss: 2.296087
Epoch: 29, Loss: 2.050281
Epoch: 30, Loss: 1.840513
Epoch: 31, Loss: 1.652652
Epoch: 32, Loss: 1.488660
Epoch: 33, Loss: 1.341410
Epoch: 34, Loss: 1.210876
Epoch: 35, Loss: 1.094708
Epoch: 36, Loss: 0.989982
Epoch: 37, Loss: 0.898018
Epoch: 38, Loss: 0.814

In [53]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [59]:
predict(model, "capital of italy?")

rome


In [56]:
list(vocab.keys())[7]

'paris'